In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
df= pd.read_csv("Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)
if "customerID" in df.columns:
  df.drop("customerID", axis=1, inplace=True)
    
categorical_cols = df.select_dtypes(include=["object"]).columns

le = LabelEncoder()
for col in categorical_cols:
  df[col] = le.fit_transform(df[col])    
X = df.drop("Churn", axis=1)
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("processing completed successfully")
print("training data shape", X_train_scaled.shape)
print("tasting data shape", X_test_scaled.shape)

processing completed successfully
training data shape (5625, 19)
tasting data shape (1407, 19)


In [1]:
import os
print(os.getcwd())

C:\Users\user


In [7]:
from sklearn.metrics import (accuracy_score,classification_report,f1_score,precision_score,recall_score,)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "SVM": SVC(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
}
results = []
for name, model in models.items():
    model.fit(X_train_scaled,y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append({"Model": name,"Accuracy": acc,"Precision": prec,"Recall": recall,"F1-Score": f1,})

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print(results_df)

                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.793888   0.624260  0.564171  0.592697
1             AdaBoost  0.786780   0.605714  0.566845  0.585635
2              XGBoost  0.773276   0.581121  0.526738  0.552595
3                  SVM  0.785359   0.626761  0.475936  0.541033
4        Random Forest  0.781805   0.615120  0.478610  0.538346
5                  KNN  0.748401   0.526455  0.532086  0.529255
6        Decision Tree  0.714286   0.465347  0.502674  0.483290


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
log_reg = LogisticRegression(random_state=42,max_iter=1000)
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100], 
    'penalty': ['l1', 'l2'], 
    'solver': ['liblinear'], 
    'class_weight': [None,'balanced',],}

grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1,)

grid_search.fit(X_train_scaled, y_train)
best_log_reg = grid_search.best_estimator_
print(' Best settings for Logistic Regression ')
print(grid_search.best_params_)
y_pred_tuned = best_log_reg.predict(X_test_scaled)
print('Classification Report')
print(classification_report(y_test, y_pred_tuned))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
 Best settings for Logistic Regression 
{'C': 10, 'class_weight': 'balanced', 'penalty': 'l1', 'solver': 'liblinear'}
Classification Report
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407



In [9]:
import joblib
joblib.dump(best_log_reg, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']